## Exploratory Data Analysis

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path.cwd().parent / 'data'

In [3]:
training_data = pd.read_csv(data_path / 'sales_data.csv')
training_data.head()

,order_id,product_id,product_category,price,is_first_time_customer,order_day_of_week,returned
0,ORD-1000,P-100,Engine,34.18,0,4,0
1,ORD-1001,P-101,Exhaust,54.07,1,0,0
2,ORD-1002,P-102,Brakes,46.28,0,6,0
3,ORD-1003,P-103,Filters,12.39,0,5,0
4,ORD-1004,P-104,Electronics,28.15,0,4,0


In [4]:
num_returned = training_data['returned'].sum()
num_not_returned = (training_data['returned'] == 0).sum()
return_rate = num_returned / num_not_returned * 100
print(f'{num_returned} returned : {num_not_returned} not returned')
print(f'{round(return_rate, 2)}% return rate')

5358 returned : 54642 not returned
9.81% return rate


In [5]:
unique_product_categories = training_data['product_category'].nunique()
unique_product_ids = training_data['product_id'].nunique()
print(f'Unique product categories: {unique_product_categories}')
print(f'Unique product IDs: {unique_product_ids}')

Unique product categories: 7
Unique product IDs: 200


In [6]:
unique_category_id_combinations = training_data.groupby(['product_category', 'product_id']).ngroups
num_in_category_id_combination = training_data.groupby(['product_category', 'product_id']).size().reset_index(name='count')
print(f'Number of possible unique combinations of categories and ids: {unique_product_categories * unique_product_ids}')
print(f'Unique combinations of categories and ids present in the dataset: {unique_category_id_combinations}')
num_in_category_id_combination.head()

Number of possible unique combinations of categories and ids: 1400
Unique combinations of categories and ids present in the dataset: 1400


,product_category,product_id,count
0,Brakes,P-100,66
1,Brakes,P-101,73
2,Brakes,P-102,66
3,Brakes,P-103,45
4,Brakes,P-104,58


In [7]:
category_codes = {
    category: code
    for code, category in enumerate(sorted(training_data['product_category'].unique()))
}
product_codes = training_data['product_id'].str.removeprefix('P-').astype(int) - 100
training_data['category_id'] = (
    training_data['product_category'].map(category_codes) * 1000 + product_codes
).astype(int)
print(category_codes)
training_data.head()

{'Brakes': 0, 'Electronics': 1, 'Engine': 2, 'Exhaust': 3, 'Filters': 4, 'HVAC': 5, 'Suspension': 6}


,order_id,product_id,product_category,price,is_first_time_customer,order_day_of_week,returned,category_id
0,ORD-1000,P-100,Engine,34.18,0,4,0,2000
1,ORD-1001,P-101,Exhaust,54.07,1,0,0,3001
2,ORD-1002,P-102,Brakes,46.28,0,6,0,2
3,ORD-1003,P-103,Filters,12.39,0,5,0,4003
4,ORD-1004,P-104,Electronics,28.15,0,4,0,1004


In [8]:
unique_prices_by_category_id = (
    training_data.groupby('category_id')['price']
    .nunique()
    .reset_index(name='unique_price_count')
)
unique_prices_by_category_id.head()

,category_id,unique_price_count
0,0,66
1,1,72
2,2,65
3,3,45
4,4,58


In [9]:
training_data['price_percentile'] = (
    training_data.groupby('category_id')['price']
    .rank(pct=True)
)
training_data.head()

,order_id,product_id,product_category,price,is_first_time_customer,order_day_of_week,returned,category_id,price_percentile
0,ORD-1000,P-100,Engine,34.18,0,4,0,2000,0.449275
1,ORD-1001,P-101,Exhaust,54.07,1,0,0,3001,0.700000
2,ORD-1002,P-102,Brakes,46.28,0,6,0,2,0.621212
3,ORD-1003,P-103,Filters,12.39,0,5,0,4003,0.083333
4,ORD-1004,P-104,Electronics,28.15,0,4,0,1004,0.392157


In [10]:
training_data.to_csv(data_path / 'training_data.csv')